In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Trỏ thẳng vào thư mục repo đã có sẵn trên Drive
%cd /content/drive/MyDrive
!git clone https://github.com/dcdube/Forecast-Models-Evaluation.git
%cd /content/drive/MyDrive/Forecast-Models-Evaluation

# 1. `model_naivedrift.py`

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib
!python models/model_naivedrift.py

# 2. `models_statsml.py`

In [ ]:
!pip install -q statsmodels pmdarima lightgbm xgboost scikit-learn
!python models/models_statsml.py

# 3. `models_neuralforecast.py`

> Models 3?9: ch?n dataset b?ng `--data` (1=B?, 2=??c, 3=London, 4=Zonnedael). V? d? l?nh b?n d??i d?ng `--data 1`.


In [ ]:
!pip install -q neuralforecast pytorch-lightning
# Truyền tham số --gpu 0 --data 1 --sampling_rate 25 để sử dụng GPU T4 trên Google Colab
!python models/models_neuralforecast.py --gpu 0 --data 1 --sampling_rate 25

# 4. `models_gluonts.py`

In [ ]:
!pip install -q gluonts ujson torch
!python models/models_gluonts.py --gpu 0 --data 1 --sampling_rate 25

# 5. `model_mamba.py`

In [ ]:
!pip install -q causal-conv1d mamba-ssm
!python models/model_mamba.py --gpu 0 --data 1 --sampling_rate 25

# 6. `fmodel_chronos.py`

In [ ]:
!pip install -q "chronos[forecasting] @ git+[https://github.com/amazon-science/chronos-forecasting.git](https://github.com/amazon-science/chronos-forecasting.git)"
!python models/fmodel_chronos.py --gpu 0 --data 1 --sampling_rate 25

# 7. `fmodel_timesfm.py`

In [ ]:
!pip install -q timesfm
!python models/fmodel_timesfm.py --gpu 0 --data 1 --sampling_rate 25

# 8. `fmodel_moirai.py`

In [ ]:
!pip install -q uni2ts
!python models/fmodel_moirai.py --gpu 0 --data 1 --sampling_rate 25

# 9. `fmodel_timerxl.py`

In [ ]:
!pip install -q transformers torch accelerate
!python models/fmodel_timerxl.py --gpu 0 --data 1 --sampling_rate 25

# 10. `fmodel_timegpt.py`

In [ ]:
!pip install -q nixtla python-dotenv
!export $(cat .env | xargs) && python models/fmodel_timegpt.py

# 13. Kiểm định ý nghĩa thống kê Diebold-Mariano (DM test) trên kết quả tự thực nghiệm
- So sánh thống kê từng cặp mô hình trên chuỗi sai số dự báo: $e_t = y_t - \hat{y}_t$
- Kiểm định giả thuyết một phía (One-sided Diebold-Mariano test):
  + $H_0$: Hai mô hình có độ chính xác dự báo tương đương ($\mathbb{E}[d_t] = 0$).
  + $H_1$: Mô hình ở trục hoành ($x$-axis) có sai số kỳ vọng nhỏ hơn có ý nghĩa thống kê so với mô hình ở trục tung ($y$-axis) ($\mathbb{E}[d_t] < 0$).
- Đánh giá đồng thời trên 2 hàm mất mát: MAE loss ($|e|$) và RMSE loss ($e^2$).
- Ngưỡng ý nghĩa: $p \le 0.05$ (vùng màu tím sẫm) khẳng định mô hình cột $x$ vượt trội có ý nghĩa thống kê; $p > 0.1$ (vùng màu vàng) thể hiện hai mô hình tương đương về mặt thống kê.
- Tự động quét toàn bộ tệp `*_forecast_vs_actual.csv` trong `results/`, căn chỉnh mốc thời gian chung và sinh bản đồ nhiệt ma trận p-value.
- Lưu trữ đồ thị chuẩn mực bài báo (PDF & PNG) tại `results/dm_test/` và hiển thị trực tiếp trên Notebook.

In [ ]:
# ==============================================================================
# 1. LỆNH CHẠY RIÊNG KIỂM ĐỊNH DIEBOLD-MARIANO LOCAL CHO CẢ 4 DATASETS:
# (Tự động tính toán & xuất toàn bộ bản đồ nhiệt PDF/PNG cho belgium, germany, london, zonnedael)
# ==============================================================================
!python utils/dm_test.py --datasets belgium germany london zonnedael --sampling all

# ==============================================================================
# 2. HOẶC CHẠY TOÀN BỘ PIPELINE (Bảng CSV + Biểu đồ cột + Kiểm định DM test cả 4 datasets):
# ==============================================================================
# !python run_all_evaluations.py


In [ ]:
# ==============================================================================
# CÁCH 2: Chạy trực tiếp Python, hiển thị bản đồ nhiệt và bảng p-value ngay trên Notebook
# ==============================================================================
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
from utils.dm_test import (
    find_model_dirs,
    read_target_series,
    _compute_common_timestamp_index,
    _align_losses_to_index,
    _build_losses_from_actual_forecast,
    build_dm_matrix
)

DM_RESULTS_DIR = Path("results/dm_test/belgium")
os.makedirs(DM_RESULTS_DIR, exist_ok=True)

# 1. Hiển thị hình ảnh ma trận nhiệt DM test trực tiếp trên notebook
heatmaps = [
    ("dm_test_pv_house_1_samp25.png", "PV House 1 (Sampling 25% - Deep Learning Models)"),
    ("dm_test_bess_house_1_samp25.png", "BESS House 1 (Sampling 25% - Deep Learning Models)"),
    ("dm_test_pv_house_1_samp100.png", "PV House 1 (Sampling 100% - Foundation Models & Baselines)")
]

for fname, title in heatmaps:
    fpath = DM_RESULTS_DIR / fname
    if fpath.exists():
        plt.figure(figsize=(12, 6))
        img = mpimg.imread(str(fpath))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Diebold-Mariano Test Heatmap: {title}", fontsize=11, fontweight="bold", pad=12)
        plt.tight_layout()
        plt.show()

# 2. Xuất bảng định lượng p-value đối chiếu các mô hình SOTA vs NaiveDrift baseline
dataset_dir = Path("results/results_belgium")
model_dirs = find_model_dirs(dataset_dir)
act_map, fc_map = {}, {}
for md in model_dirs:
    a, f, _ = read_target_series(md, "PV_house_1_forecast_vs_actual.csv", "Sampling_25")
    if f is not None:
        act_map[md.name] = a
        fc_map[md.name] = f

c_idx = _compute_common_timestamp_index(fc_map)
act_map = _align_losses_to_index(act_map, c_idx)
fc_map = _align_losses_to_index(fc_map, c_idx)
mae_losses, rmse_losses = {}, {}
models = list(fc_map.keys())
for m in models:
    mae_losses[m], rmse_losses[m] = _build_losses_from_actual_forecast(act_map[m], fc_map[m])

df_mae = pd.DataFrame(build_dm_matrix(mae_losses, models), index=models, columns=models)
df_rmse = pd.DataFrame(build_dm_matrix(rmse_losses, models), index=models, columns=models)

summary_rows = []
for m in models:
    if m == "NaiveDrift": continue
    p_mae = df_mae.loc[m, "NaiveDrift"]
    p_rmse = df_rmse.loc[m, "NaiveDrift"]
    summary_rows.append({
        "Mô hình": m,
        "p-value (MAE loss)": f"{p_mae:.6f}",
        "p-value (RMSE loss)": f"{p_rmse:.6f}",
        "Ý nghĩa thống kê (alpha=0.05)": "Vượt trội có ý nghĩa (p < 0.05)" if p_mae <= 0.05 else "Tương đương (p > 0.05)"
    })

df_summary = pd.DataFrame(summary_rows)
print("\n" + "="*85)
print("BẢNG TỔNG HỢP KIỂM ĐỊNH DIEBOLD-MARIANO TỰ THỰC NGHIỆM (PV 1, Sampling 25%)")
print("Giả thuyết đối H1: Mô hình kiểm định có sai số thấp hơn NaiveDrift baseline")
print("="*85)
try:
    from IPython.display import display
    display(df_summary)
except Exception:
    print(df_summary.to_string(index=False))
